[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lucascamillomd/pyaging/blob/main/tutorials/tutorial_dnam.ipynb) [![Open In nbviewer](https://img.shields.io/badge/View%20in-nbviewer-orange)](https://nbviewer.jupyter.org/github/lucascamillomd/pyaging/blob/main/tutorials/tutorial_dnam.ipynb)

# Illumina Human Methylation Arrays

This tutorial is a brief guide for the implementation of an array of bulk DNA-methylation epigenetic clocks that predict age in humans. In this notebook, we will demonstrate the breadth of epigenetic clock models available in `pyaging` by showing:

- Horvath's 2013 ElasticNet-based clock ([paper](https://genomebiology.biomedcentral.com/articles/10.1186/gb-2013-14-10-r115));
  
- AltumAge, a highly accurate deep-learning based clock ([paper](https://www.nature.com/articles/s41514-022-00085-y));
    
- PCGrimAge, a principal-component based version of the GrimAge clock ([paper](https://www.nature.com/articles/s43587-022-00248-2));

- GrimAge2, the latest version of GrimAge ([paper](https://www.aging-us.com/article/204434/text]));

- DunedinPACE, a biomarker of the pace of aging ([paper](https://elifesciences.org/articles/73420)).

We just need two packages for this tutorial.

In [1]:
import pandas as pd
import pyaging as pya

## Download and load example data

Let's download the publicly avaiable dataset GSE139307 with Illumina's 450k array. The CpG coverage of the 450k array should be good enough for most clocks.

In [2]:
pya.data.download_example_data('GSE139307')

⏺ example data already at pyaging_data/GSE139307.pkl

'pyaging_data/GSE139307.pkl'

In [3]:
df = pd.read_pickle('pyaging_data/GSE139307.pkl')

In [4]:
df.head()

,dataset,tissue_type,age,gender,cg00000029,cg00000108,cg00000109,cg00000165,cg00000236,cg00000289,...,ch.X.93511680F,ch.X.938089F,ch.X.94051109R,ch.X.94260649R,ch.X.967194F,ch.X.97129969R,ch.X.97133160R,ch.X.97651759F,ch.X.97737721F,ch.X.98007042R
GSM4137709,GSE139307,sperm,84.0,M,0.084811,0.920696,0.856851,0.084567,0.838699,0.247273,...,0.061751,0.045942,0.037631,0.056455,0.249872,0.049022,0.085691,0.037435,0.077820,0.106234
GSM4137710,GSE139307,sperm,69.0,M,0.099626,0.919073,0.890024,0.115541,0.852584,0.198103,...,0.075077,0.041849,0.032573,0.089790,0.250245,0.079095,0.079756,0.046229,0.091256,0.120241
GSM4137711,GSE139307,sperm,69.0,M,0.117228,0.920276,0.894317,0.117127,0.839258,0.213410,...,0.068679,0.049515,0.058097,0.079919,0.299758,0.079305,0.089815,0.065364,0.086864,0.156005
GSM4137712,GSE139307,sperm,69.0,M,0.077096,0.910204,0.908400,0.073885,0.861615,0.163276,...,0.070091,0.033289,0.038836,0.108213,0.295428,0.050731,0.099943,0.047597,0.078480,0.107480
GSM4137713,GSE139307,sperm,67.0,M,0.063524,0.911608,0.884643,0.079877,0.864654,0.176169,...,0.082368,0.038411,0.048787,0.088631,0.316694,0.041873,0.079303,0.048823,0.089010,0.117903


For PCGrimAge and GrimAge2, both age and sex are features. Therefore, to get the full prediction, let's convert the column `gender` into a column called `female`, with 1 being female and 0 being male.

In [5]:
# needs only numerical data (doesn't work with strings)
df['female'] = (df['gender'] == 'F').astype(int)

Moreover, it is important to note that some probes are duplicated in the EPICv2 array, following the format cg#########_BC11 and cg#########_TC11 for the opposite strands. Given that at this moment most clocks have not been trained with EPICv2 data directly, it is recommended to average these probes. This is particularly the case for DunedinPACE, from which some clock probes were duplicated in the update from EPICv1. To remedy this issue, simply use the following function to aggregate any duplicated probes that may be present.

In [6]:
df = pya.pp.epicv2_probe_aggregation(df)

## Convert data to AnnData object

AnnData objects are highly flexible and are thus our preferred method of organizing data for age prediction.

In [7]:
adata = pya.pp.df_to_adata(df, metadata_cols=['gender', 'tissue_type', 'dataset'], imputer_strategy='knn')

Note that the original DataFrame is stored in `X_original` under layers. is This is what the `adata` object looks like:

In [8]:
adata

AnnData object with n_obs × n_vars = 37 × 485513
    obs: 'gender', 'tissue_type', 'dataset'
    var: 'percent_na'
    uns: 'imputer_strategy'
    layers: 'X_original', None (.X), 'X_imputed'

## Predict age

We can either predict one clock at once or all at the same time. For convenience, let's simply input all four clocks of interest at once. The function is invariant to the capitalization of the clock name. 

In [9]:
pya.pred.predict_age(adata, ['Horvath2013', 'AltumAge', 'PCGrimAge', 'GrimAge2', 'DunedinPACE'])

In [10]:
adata.obs.head()

,gender,tissue_type,dataset,horvath2013,altumage,pcgrimage,grimage2,dunedinpace
GSM4137709,M,sperm,GSE139307,33.624776,37.007213,95.506114,77.581057,1.326327
GSM4137710,M,sperm,GSE139307,28.829344,29.426899,83.934244,65.926346,1.215611
GSM4137711,M,sperm,GSE139307,28.316545,22.798928,82.709334,63.358341,1.271091
GSM4137712,M,sperm,GSE139307,24.850630,18.079173,84.269462,60.218880,1.276866
GSM4137713,M,sperm,GSE139307,25.942111,20.071985,84.356985,61.235919,1.262023


For curiosity, we can also check if there are any correlations amongst these clocks.

In [11]:
adata.obs.iloc[:, 3:].corr('pearson')

,horvath2013,altumage,pcgrimage,grimage2,dunedinpace
horvath2013,1.000000,0.676242,0.211881,0.459193,0.354771
altumage,0.676242,1.000000,0.156456,0.440044,0.164102
pcgrimage,0.211881,0.156456,1.000000,0.859490,0.061486
grimage2,0.459193,0.440044,0.859490,1.000000,0.183721
dunedinpace,0.354771,0.164102,0.061486,0.183721,1.000000


After age prediction, the clocks are added to `adata.obs`. Moreover, the percent of missing values for each clock and other metadata are included in `adata.uns`.

In [12]:
adata

AnnData object with n_obs × n_vars = 37 × 485513
    obs: 'gender', 'tissue_type', 'dataset', 'horvath2013', 'altumage', 'pcgrimage', 'grimage2', 'dunedinpace'
    var: 'percent_na'
    uns: 'imputer_strategy', 'horvath2013_percent_na', 'horvath2013_missing_features', 'horvath2013_metadata', 'altumage_percent_na', 'altumage_missing_features', 'altumage_metadata', 'pcgrimage_percent_na', 'pcgrimage_missing_features', 'pcgrimage_metadata', 'grimage2_percent_na', 'grimage2_missing_features', 'grimage2_metadata', 'dunedinpace_percent_na', 'dunedinpace_missing_features', 'dunedinpace_metadata'
    layers: 'X_original', None (.X), 'X_imputed'

We can also look at which features seem to be missing from each clock (if there are any).

In [13]:
adata.uns['dunedinpace_missing_features']

[]

## Get citation

The doi, citation, and some metadata are automatically added to the AnnData object under `adata.uns[CLOCKNAME_metadata]`.

In [14]:
adata.uns['horvath2013_metadata']

{'clock_name': 'horvath2013',
 'data_type': 'DNA methylation',
 'species': 'Homo sapiens',
 'year': 2013,
 'approved_by_author': '⌛',
 'citation': 'Horvath, S. DNA methylation age of human tissues and cell types. Genome Biology 14, R115 (2013).',
 'doi': 'https://doi.org/10.1186/gb-2013-14-10-r115',
 'notes': 'Pan-tissue DNAm-age predictor fitted by elastic net to a transformed chronological-age outcome and returned to the year scale; it uses 353 CpGs shared between the 27K and 450K arrays.',
 'research_only': None,
 'tissue': ['multi-tissue'],
 'predicts': ['chronological age'],
 'training_target': ['chronological age'],
 'unit': ['years'],
 'model_type': 'elastic net regression',
 'platform': ['Illumina 27K', 'Illumina 450K'],
 'population': 'all ages',
 'journal': 'Genome Biology',
 'last_author': 'Steve Horvath',
 'n_features': 353,
 'citations': 7318,
 'citations_date': '2026-07-05',
 'version': 'v0.3.0',
 'postprocess': 'anti_log_linear',
 'reference_values': True}

In [15]:
adata.uns['altumage_metadata']

{'clock_name': 'altumage',
 'data_type': 'DNA methylation',
 'species': 'Homo sapiens',
 'year': 2022,
 'approved_by_author': '✅',
 'citation': 'de Lima Camillo, L.P., Lapierre, L.R. & Singh, R. A pan-tissue DNA-methylation epigenetic clock based on deep learning. npj Aging 8, 4 (2022).',
 'doi': 'https://doi.org/10.1038/s41514-022-00085-y',
 'notes': 'Pan-tissue chronological-age predictor using a five-hidden-layer neural network and 20,318 CpGs shared across the 27K, 450K and EPIC manifests; the actual training data came from 27K and 450K datasets.',
 'research_only': None,
 'tissue': ['multi-tissue'],
 'predicts': ['chronological age'],
 'training_target': ['chronological age'],
 'unit': ['years'],
 'model_type': 'deep neural network',
 'platform': ['Illumina 27K', 'Illumina 450K'],
 'population': 'all ages',
 'journal': 'npj Aging',
 'last_author': 'Ritambhara Singh',
 'n_features': 20318,
 'citations': 145,
 'citations_date': '2026-07-05',
 'version': 'v0.3.0',
 'preprocess': 'sca

In [16]:
adata.uns['pcgrimage_metadata']

{'clock_name': 'pcgrimage',
 'data_type': 'DNA methylation',
 'species': 'Homo sapiens',
 'year': 2022,
 'approved_by_author': '⌛',
 'citation': 'Higgins-Chen, Albert T., et al. "A computational solution for bolstering reliability of epigenetic clocks: implications for clinical trials and longitudinal tracking." Nature Aging 2 (2022): 644–661.',
 'doi': 'https://doi.org/10.1038/s43587-022-00248-2',
 'notes': 'Principal-component proxy trained to reproduce the original DNAm GrimAge score; age and sex are additional model inputs.',
 'research_only': None,
 'tissue': ['whole blood'],
 'predicts': ['mortality risk'],
 'training_target': ['DNAm GrimAge output'],
 'unit': ['years'],
 'model_type': 'PCA + elastic net regression',
 'platform': ['Illumina 450K'],
 'population': 'adults',
 'journal': 'Nature Aging',
 'last_author': 'Morgan E. Levine',
 'n_features': 78466,
 'citations': 497,
 'citations_date': '2026-07-05',
 'version': 'v0.3.0',
 'reference_values': True}

In [17]:
adata.uns['grimage2_metadata']

{'clock_name': 'grimage2',
 'data_type': 'DNA methylation',
 'species': 'Homo sapiens',
 'year': 2022,
 'approved_by_author': '⌛',
 'citation': 'Lu, A. T., et al. “DNA methylation GrimAge version 2.” Aging 14(23): 9484–9549 (2022).',
 'doi': 'https://doi.org/10.18632/aging.204434',
 'notes': 'Mortality-risk epigenetic clock combining ten blood DNAm surrogate biomarkers with chronological age and sex; the Cox linear predictor is calibrated to an age-like value in years.',
 'research_only': True,
 'tissue': ['whole blood'],
 'predicts': ['mortality risk'],
 'training_target': ['mortality'],
 'unit': ['years'],
 'model_type': 'elastic net Cox regression',
 'platform': ['Illumina 450K'],
 'population': 'older adults',
 'journal': 'Aging',
 'last_author': 'Steve Horvath',
 'n_features': 1032,
 'citations': 291,
 'citations_date': '2026-07-05',
 'version': 'v0.3.0',
 'postprocess': 'cox_to_years',
 'reference_values': True}

In [18]:
adata.uns['dunedinpace_metadata']

{'clock_name': 'dunedinpace',
 'data_type': 'DNA methylation',
 'species': 'Homo sapiens',
 'year': 2022,
 'approved_by_author': '✅',
 'citation': 'Belsky, D. W., Caspi, A., Corcoran, D. L., et al. (2022). DunedinPACE, a DNA methylation biomarker of the pace of aging. eLife, 11, e73420.',
 'doi': 'https://doi.org/10.7554/elife.73420',
 'notes': 'Whole-blood elastic-net pace-of-aging biomarker trained at age 45 against a 20-year longitudinal slope composite of 19 organ-system biomarkers. PyAging follows the official 20,000-probe quantile-normalization panel: 173 scoring CpGs plus 19,827 background probes.',
 'research_only': True,
 'tissue': ['whole blood'],
 'predicts': ['pace of aging'],
 'training_target': ['pace of aging'],
 'unit': ['biological years per chronological year'],
 'model_type': 'elastic net regression',
 'platform': ['Illumina EPIC'],
 'population': 'adults',
 'journal': 'eLife',
 'last_author': 'Terrie E. Moffitt',
 'n_features': 20000,
 'citations': 967,
 'citations_